
#### OpenAI Tool Call Demo with Real APIs (Tomorrow.io + Geocode)

- This Jupyter Notebook demonstrates how a Large Language Model (LLM) can reason about a user query,
- decide when to invoke external tools (APIs), and integrate real-world information back into its response.

https://docs.tomorrow.io/reference/api-authentication

---

In [19]:
#pip intall timezonefinder

In [20]:
# pip install langchain-community

In [49]:
import openai
import json
import requests
from datetime import datetime
import pytz
import os

from dotenv import load_dotenv

In [50]:
# Load API keys from environment variables
load_dotenv()

openai.api_key      = os.getenv("OPENAI_API_KEY")
TOMORROW_IO_API_KEY = os.getenv("TOMORROW_IO_API_KEY")
GEOCODE_API_KEY     = os.getenv("GEOCODE_API_KEY")

In [51]:
# --------------------------------------------------
# Utility: Geocode a city name to lat/lon using Maps.co
# --------------------------------------------------
def geocode_city(city: str):
    url = f"https://geocode.maps.co/search?q={city}"
    res = requests.get(url)
    try:
        res.raise_for_status()
    except requests.exceptions.HTTPError as e:
        raise ValueError(f"Geocoding API error {res.status_code}: {res.text}") from e
    data = res.json()
    if isinstance(data, list) and len(data) > 0:
        lat = float(data[0]['lat'])
        lon = float(data[0]['lon'])
        return lat, lon
    else:
        raise ValueError(f"Geocoding failed for '{city}' — no results found.")

In [52]:
geocode_city("Kolkata")

(22.5726459, 88.3638953)

In [53]:
# --------------------------------------------------
# Get weather using Tomorrow.io API
# --------------------------------------------------
def get_weather(location: str) -> str:
    lat, lon = geocode_city(location)
    
    url = f"https://api.tomorrow.io/v4/weather/realtime?location={lat},{lon}&apikey={TOMORROW_IO_API_KEY}"
    
    res = requests.get(url)
    
    if res.status_code != 200:
        raise ValueError(f"Tomorrow.io API error: {res.status_code}")
        
    data = res.json()
    values    = data.get("data", {}).get("values", {})
    temp      = values.get("temperature")
    condition = values.get("weatherCode", "unknown")
    
    return f"In {location}, it's {temp}°C with condition code '{condition}'."

In [54]:
# --------------------------------------------------
# Get local time based on lat/lon from geocode
# --------------------------------------------------
def get_time(city: str) -> str:
    lat, lon = geocode_city(city)
    from timezonefinder import TimezoneFinder
    tf = TimezoneFinder()
    tz_str = tf.timezone_at(lat=lat, lng=lon)
    if not tz_str:
        raise ValueError(f"Could not determine timezone for {city}")
    now = datetime.now(pytz.timezone(tz_str)).strftime("%Y-%m-%d %H:%M:%S")
    return f"The current time in {city} is {now} ({tz_str})."

In [55]:
# --------------------------------------------------
# Define tool (function) specifications
# --------------------------------------------------
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given location using Tomorrow.io.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name"}
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_time",
            "description": "Get the current time in a city using geocode and timezone info.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"}
                },
                "required": ["city"]
            }
        }
    }
]


In [56]:
# --------------------------------------------------
# User query and chat initiation
# --------------------------------------------------
# messages = [
#     {"role": "user", "content": "What's the weather in Mumbai and the time in Berlin?"}
# ]

messages = [
    {"role": "user", "content": "What's the weather in Delhi and Bangalore? compare the temperatures there"}
]

messages = [
    {"role": "user", "content": "would it be ok to travel to Singapore vs Delhi. from climate point of view. "}
]

messages = [
    {"role": "user", "content": "can I be under a tree in Delhi at 3pm afternoon?"}
]

# messages = [
#     {"role": "user", "content": "What's the weather in Mumbai and the time in Berlin?"}
# ]


In [57]:
# Initialize OpenAI client
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [58]:
response = client.chat.completions.create(
    model      = "gpt-4o-mini",
    messages   = messages,
    tools      = tools,
    tool_choice= "auto"
)

In [59]:
# Check and parse tool calls
tool_messages = []

for tool_call in response.choices[0].message.tool_calls:
    fn_name = tool_call.function.name
    fn_args = json.loads(tool_call.function.arguments)

    try:
        if fn_name == "get_weather":
            result = get_weather(**fn_args)
        elif fn_name == "get_time":
            result = get_time(**fn_args)
        else:
            result = "Unknown tool"
    except Exception as e:
        result = f"Error calling {fn_name}: {str(e)}"

    tool_messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result
    })


TypeError: 'NoneType' object is not iterable

In [ ]:
# --------------------------------------------------
# Feed tool results back to model
# --------------------------------------------------
final_messages = messages + [response.choices[0].message] + tool_messages

final_response = client.chat.completions.create(
    model   = "gpt-4o-mini",
    messages= final_messages
)

In [34]:
# --------------------------------------------------
# Final LLM response using tool results
# --------------------------------------------------
print("\nFinal Answer:")
print(final_response.choices[0].message.content)


Final Answer:
From a climate perspective, currently:

- **Singapore**: The temperature is about 30.5°C with relatively stable weather conditions.
- **Delhi**: The temperature is around 32.4°C, and it may be more prone to pollution and heat issues, especially during certain times of the year.

Overall, Singapore generally has a more humid tropical climate, while Delhi experiences hot summers with potential air quality concerns. Depending on your tolerance for heat and humidity, you might find Singapore's climate more comfortable at this time. If the weather is a significant factor for your travel decision, Singapore may be the better option.


#### Significance of LLM Here:


1. **Natural Language Understanding**:

The LLM reads a user message like `What's the weather in Mumbai and the time in Berlin?` and breaks it down into two sub-tasks: fetch weather for Mumbai and time for Berlin.

2. **Tool Selection and Argument Construction**:

Based on its training and system prompt/tool schema, the LLM identifies the right tools to call (e.g., `get_weather` and `get_time`) and automatically formats the correct input arguments:
- `get_weather({"location": "Mumbai"})`
- `get_time({"city": "Berlin"})`

3. **Delegation to External Systems**:

The LLM doesn't know the real-time weather or current time, so it delegates those queries to APIs via tool calls.

4. **Final Answer Synthesis**:

Once tool results are returned (like temperature or local time), the LLM integrates them back into a complete, natural-language response like:
      - "In Mumbai, it's 29°C and in Berlin the current time is 7:20 PM (Europe/Berlin)."

5. **Automation Without Hard-Coding**:

You don’t have to write separate logic for parsing input, choosing APIs, forming responses, or handling multiple queries — the LLM handles this flexibly.

This approach is scalable: if you add more tools (e.g., for stock prices, flight status, health data), the LLM can dynamically decide when and how to use them — all via natural conversation.


----
#### Using Langchain with OpenAI Tool Calls
---

In [17]:
from langchain.tools import tool
from langchain.agents import initialize_agent, AgentType
from langchain.llms import OpenAI
import requests
from datetime import datetime
import pytz
from dotenv import load_dotenv
import os


In [19]:
# Load API keys
load_dotenv()
TOMORROW_IO_API_KEY = os.getenv("TOMORROW_IO_API_KEY")
openai.api_key      = os.getenv("OPENAI_API_KEY")
TOMORROW_IO_API_KEY = os.getenv("TOMORROW_IO_API_KEY")
GEOCODE_API_KEY     = os.getenv("GEOCODE_API_KEY")

In [20]:
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location using Tomorrow.io."""
    url = f"https://geocode.maps.co/search?q={location}"
    res = requests.get(url)
    res.raise_for_status()
    data = res.json()
    if isinstance(data, list) and len(data) > 0:
        lat = float(data[0]['lat'])
        lon = float(data[0]['lon'])
    else:
        return f"Geocoding failed for '{location}' — no results found."
    url = f"https://api.tomorrow.io/v4/weather/realtime?location={lat},{lon}&apikey={TOMORROW_IO_API_KEY}"
    res = requests.get(url)
    if res.status_code != 200:
        return f"Tomorrow.io API error: {res.status_code}"
    data = res.json()
    values = data.get("data", {}).get("values", {})
    temp = values.get("temperature")
    condition = values.get("weatherCode", "unknown")
    return f"In {location}, it's {temp}°C with condition code '{condition}'."

In [21]:
@tool
def get_time(city: str) -> str:
    """Get the current time in a city using geocode and timezone info."""
    url = f"https://geocode.maps.co/search?q={city}"
    res = requests.get(url)
    res.raise_for_status()
    data = res.json()
    if isinstance(data, list) and len(data) > 0:
        lat = float(data[0]['lat'])
        lon = float(data[0]['lon'])
    else:
        return f"Geocoding failed for '{city}' — no results found."
    from timezonefinder import TimezoneFinder
    tf = TimezoneFinder()
    tz_str = tf.timezone_at(lat=lat, lng=lon)
    if not tz_str:
        return f"Could not determine timezone for {city}"
    now = datetime.now(pytz.timezone(tz_str)).strftime("%Y-%m-%d %H:%M:%S")
    return f"The current time in {city} is {now} ({tz_str})."

In [25]:
# Initialize LLM
llm = OpenAI(temperature=0)

In [26]:
# Create agent with tools
agent = initialize_agent(
    tools=[get_weather, get_time],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [27]:
# Example query
result = agent.run("What's the weather in Delhi and the time in Berlin?")
print(result)



> Entering new AgentExecutor chain...
 I need to get the weather for Delhi and the time for Berlin.
Action: get_weather
Action Input: Delhi
Observation: In Delhi, it's 26.9°C with condition code '1102'.
Thought: I now need to get the time for Berlin.
Action: get_time
Action Input: Berlin
Observation: The current time in Berlin is 2025-09-11 21:37:30 (Europe/Berlin).
Thought: I now know the final answer.
Final Answer: In Delhi, it's 26.9°C with condition code '1102' and the current time in Berlin is 2025-09-11 21:37:30 (Europe/Berlin).

> Finished chain.
In Delhi, it's 26.9°C with condition code '1102' and the current time in Berlin is 2025-09-11 21:37:30 (Europe/Berlin).
